# ChordVision Web App Server Control

Use this notebook to **start**, **check status**, and **stop** the FastAPI web server for the Chord Visualizer.

> **Kernel**: Make sure the kernel is set to **`Python (seperate)`**.

## 1. Helper Server Manager Class
Run this cell once to define the background server controller.

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path and set as current working directory
PROJECT_ROOT = Path("..").resolve() if Path("..", "module").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import os
import sys
import time
import socket
import subprocess
import urllib.request
import json

SERVER_PORT = 8080
SERVER_PROCESS = None

def is_port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("0.0.0.0", port)) == 0

def start_server(port=SERVER_PORT):
    global SERVER_PROCESS
    if is_port_in_use(port):
        print(f"⚠️  Port {port} is already in use! (Server is already running)")
        print(f"👉 Open Web App: http://localhost:{port}")
        return
    
    python_bin = sys.executable
    cmd = [python_bin, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", str(port), "--reload"]
    
    SERVER_PROCESS = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        cwd=str(PROJECT_ROOT)
    )
    
    # Wait up to 3 seconds for server to start
    for _ in range(15):
        time.sleep(0.2)
        if is_port_in_use(port):
            break
            
    if is_port_in_use(port):
        print(f"🚀 Server successfully started on PID {SERVER_PROCESS.pid}!")
        print(f"🌐 Web App URL: http://localhost:{port}")
    else:
        print("❌ Failed to start server. Checking output:")
        if SERVER_PROCESS.poll() is not None:
            out, _ = SERVER_PROCESS.communicate()
            print(out)

def stop_server(port=SERVER_PORT):
    global SERVER_PROCESS
    killed = False
    
    # 1. Terminate tracked subprocess
    if SERVER_PROCESS and SERVER_PROCESS.poll() is None:
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=2)
        except subprocess.TimeoutExpired:
            SERVER_PROCESS.kill()
        SERVER_PROCESS = None
        killed = True
        
    # 2. Kill any remaining processes occupying the port
    try:
        res = subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True, text=True)
        if res.returncode == 0:
            killed = True
    except Exception:
        try:
            res = subprocess.run(f"lsof -ti :{port} | xargs kill -9", shell=True, capture_output=True)
            if res.returncode == 0:
                killed = True
        except Exception:
            pass
            
    time.sleep(0.5)
    if not is_port_in_use(port):
        print(f"🛑 Server on port {port} has been completely STOPPED.")
    else:
        print(f"⚠️  Port {port} still appears active. Please retry stopping.")

def check_status(port=SERVER_PORT):
    if not is_port_in_use(port):
        print(f"🔴 Server is NOT running (Port {port} is closed).")
        return
        
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}/api/stems", timeout=2) as resp:
            data = json.loads(resp.read().decode())
            stems = [s['name'] for s in data.get('stems', [])]
            print(f"🟢 Server is HEALTHY and RUNNING on http://localhost:{port}")
            print(f"   Available Stems ({len(stems)}): {', '.join(stems)}")
    except Exception as e:
        print(f"🟡 Server port is open, but API returned error: {e}")


## 2. Start the Server
Run this cell to launch the FastAPI web application in the background.

In [2]:
start_server()

❌ Failed to start server. Checking output:


## 3. Check Server Status & Available Audio Tracks
Run this cell to verify server health and inspect available audio tracks in `data/`.

In [3]:
check_status()

🔴 Server is NOT running (Port 8080 is closed).


## 4. Stop the Server
Run this cell whenever you want to stop the web app.

In [4]:
stop_server()

🛑 Server on port 8080 has been completely STOPPED.
